<a href="https://colab.research.google.com/github/Deepesh-Kumar-Singh/LLM_From_scratch/blob/main/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

In [2]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
x=torch.tensor([[1,5,6,3,4,4,5,8],[3,5,2,4,5,7,]])


cpu


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self,embed_size,heads):
        super(SelfAttention,self).__init__()
        self.embed_size=embed_size
        self.heads=heads
        self.head_dim=embed_size//heads

        assert (self.head_dim*heads==embed_size),"Embed size needs to be div by heads"
        self.values=nn.Linear(embed_size,embed_size,bias=False) #input:x, Linear(:wx,)
        self.keys=nn.Linear(embed_size,embed_size,bias=False)
        self.queries=nn.Linear(embed_size,embed_size,bias=False)
    def forward(self,values,keys,query,mask=None):
      #query.shape=(N,query_len,embed_size)
      N=query.shape[0]
      value_len,key_len,query_len=values.shape[1],keys.shape[1],query.shape[1]
      values=self.values(values)#(N,value_len,embed_size)
      keys=self.keys(keys) #(N,key_len,embed_size)
      queries=self.queries(query)#(N,query_len,embed_size)

      #split the empbeding into self.heads piece

      values=values.reshape(N,value_len,self.heads,self.head_dim) #self.head*self.head_dim=self.embes size
      keys=keys.reshape(N,key_len,self.heads,self.head_dim)
      queries=queries.reshape(N,query_len,self.heads,self.head_dim)
      #queries shape=(N,query_len,head,heads_dim)-> nqhd
      #keys shape=(N,keys_len,head,heads_dim)-> nkhd
      #attn_scores shape=(N,head,query_len,key_len)-> nhqk
      attn_scores=torch.einsum("nqhd,nkhd->nhqk",[queries,keys])
      #Now we create a mask as we know that length of queries is not same therefore we add padding
      if mask is not None:
        attn_scores=attn_scores.masked_fill(mask==0,float("-1e20"))
      #Now according to formula scale and normalize
      attention=torch.softmax(attn_scores/(self.head_dim**(1/2)),dim=3)
      #attention_shape:(N,heads,query_len,key_len)->nhql
      #values shape:(N,value_len,heads,heads_dim)-nlhd
      #Now again reshape and concetenate the two dimension.
      out=torch.einsum("nhql,nlhd->nqhd",[attention,value]).reshape(N,query_len,self.heads*self.head_dim)
      #we reshape it so that we can merge the hd and get one v
      #print(f"out attn_scores shape:{out.shape}")
      out=self.fc_out(out) #(N,query_len,embed_size)
      return out



In [ ]:
#Now we have the multihead atteention and now we will make add and norms and feed forward
class TransformerBlock(nn.Module):
  def __init__(self,embed_size,heads,dropout,forward_expansion):
    super(TransformerBlock,self).__init__()
    self.attention=SelfAttention(embed_size,heads)
    self.norm1=nn.LayerNorm(embed_size)
    self.norm2=nn.LayerNorm(embed_size)
    self.feed_forward=nn.Sequential(
        nn.Linear(embed_size,forward_expansion*embed_size),
        nn.ReLU(),
        nn.Linear(forward_expansion*embed_size,embed_size)
    )
    self.dropout=nn.Dropout(dropout)
  def forward(self,value,key,query,mask):
    attention=self.attention(value,key,query,mask)

    x=self.dropout(self.norm1(attention+query))
    forward=self.feed_forward(x)
    out=self.dropout(self.norm2(forward+x))
    return out



In [ ]:
class Encoder(nn.Module):
  def __init__(self,src_vocab_size,embed_size,num_layers,heads,device,forward_expansion,dropout,max_length):
    super(Encoder,self).__init__()
    self.embed_size=embed_size
    self.device=device
    self.word_embedding=nn.Embedding(src_vocab_size,embed_size)
    self.position_embedding=nn.Embedding(max_length,embed_size)
    self.layers=nn.ModuleList(
        [
         TransformerBlock(
             embed_size,
             heads,
             dropout=dropout,
             forward_expansion=forward_expansion
         )
        for _ in range(num_layers)
        ])
    self.dropout=nn.Dropout(dropout)
  def forward(self,x,mask):
    N,seq_length=x.shape
    positions=torch.arange(0,seq_length).expand(N,seq_length).to(self.device)
    out=self.dropout(self.word_embedding(x)+self.position_embedding(positions))
    for layer in self.layers:
      out=layer(out,out,out,mask)
    return out


In [ ]:
class DecoderBlock(nn.Module):
  def __init__(self,embed_size,num_layers,heads,forward_expansion,dropout,device):
    super(DecoderBlock,self).__init__()
    self.norm=nn.LayerNorm(embed_size)
    self.attention=SelfAttention(embed_size,heads=heads)
    self.transformer_block=TransformerBlock(
        embed_size,heads,dropout,forward_expansion
    )
    self.dropout=nn.Dropout(dropout)

  def forward(self,x,value,key,src_mask,trg_mask):
    attention=self.attention(x,x,x,trg_mask)
    query=self.dropout(self.norm(attention+x))
    out=self.transformer_block(value,key,query,src_mask)
    return out

In [ ]:
class Decoder(nn.Module):
  def __init__(self,trg_vocab_size,embed_size,num_layers,heads,forward_expansion,dropout,device,max_length):
    super(Decoder,self).__init__()
    self.device=device
    self.word_embedding=nn.Embedding(trg_vocab_size,embed_size)
    self.position_embedding=nn.Embedding(max_length,embed_size)
    self.layers=nn.ModuleList(
        [
            DecoderBlock(embed_size,heads,forward_expansion,dropout,device)
            for _ in range(num_layers)
        ]

    )
    self.fc_out=nn.Linear(embed_size,trg_vocab_size) #
    self.dropout=nn.Dropout(dropout)
  def forward(self,x,enc_out,src_mask,trg_mask):
    N,seq_length=x.shape
    positions=torch.arange(0,seq_length).expand(N,seq_length).to(self.device)
    x=self.dropout(self.word_embedding(x)+self.position_embedding(positions))
    for layer in self.layers:
      x=layer(x,enc_out,enc_out,src_mask,trg_mask)
    out=self.fc_out(x)
    return out

In [ ]:
class Transformer(nn.Module):
  def __init__(self,src_vocab_size,trg_vocab_size,src_pad_idx,trg_pad_idx,embed_size=512,num_layers=6,forward_expansion=4,heads=8,dropout=0,max_length=100)
    super(Transformer,self).__init__()
    self.encoder=Encoder(
      src_vocab_size,
      embed_size,
      num_layers,
      heads,
      device,
      forward_expansion,
      dropout,
      max_length,
    )
    self.decoder=Decoder(
      trg_vocab_size,
      embed_size,
      num_layers,
      heads,
      forward_expansion,
      dropout,
      device,
      max_length,
    )
    self.src_pad_idx=src_pad_idx #pad token
    self.trg_pad_idx=trg_pad_idx #future token mask
    self.device=device
  def make_src_mask(self,src):
    src_mask=(src!=self.src_pad_idx).unsqueeze(1).unsqueeze(2)
    #(N,1,1,src_len)
    return src_mask.to(self.device)
  def make_trg_mask(self,trg):
    N,trg_len=trg.shape
    trg_mask=torch.tril(torch.ones((trg_len,trg_len))).expand(
        N,1,trg_len,trg_len)
    return trg_mask.to(self.device)
  def forward(self,src,trg):
    src_mask=self.make_src_mask(src)
    trg_mask=self.make_trg_mask(trg)
    enc_src=self.encoder(src,src_mask)
    out=self.decoder(trg,enc_src,src_mask,trg_mask)
    return out